# hh_lab: 25 вопросов о рынке IT-труда

В базе вакансии hh.ru за 2019 и 2020 годы, всего 1 123 593 строки. Вопросы идут от простых выборок к оконным функциям и работе с производительностью.



In [33]:
import getpass
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_colwidth", 80)
engine = create_engine(f"postgresql+psycopg2://{getpass.getuser()}@localhost:5433/hh_lab")

def q(sql, **params):
    """Выполнить SELECT и вернуть DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

def run(sql):
    """Выполнить команду, которая меняет базу (CREATE VIEW и т. п.)."""
    with engine.begin() as conn:
        conn.execute(text(sql))

## Блок 1. Основная таблица и справочники

Темы: GROUP BY, FILTER, CASE, доли и медианы.

### 1. Сколько вакансий опубликовано в каждом году и какая доля из них с зарплатой?

- **Таблицы:** vacancies



In [7]:
# Вопрос 1: q("""SELECT ...""") 
q(""" 
SELECT extract(year FROM published_at) ::int AS year,
count(*) as vacantion,
count(*) FILTER (WHERE salary_from IS NOT NULL OR salary_to IS NOT NULL) AS with_salary,
round(100 * count(*) filter(WHERE salary_from IS NOT NULL OR salary_to IS NOT NULL) / count(*),1) as with_salary_pct
FROM vacancies                                                                                  
GROUP BY year
Order by year;
""")

,year,vacantion,with_salary,with_salary_pct
0,2019,535956,266918,49.0
1,2020,587637,285014,48.0


**Вывод:** 
49 процентов вакансий для 2019 года и 48 для 2020 имеют одну из границ заработной платы, от года к году этот показатель оказался примерно равным. 
За исключением того что в 2020 году вакансий стало больше а процент с указанием зп стал ниже можем сказать что в 2020 году мы заметили тенденцию согласовывать заработную плату при собеседовании.


### 2. Как менялась доля каждого графика работы между 2019 и 2020 годами? Какой график вырос сильнее всего?

- **Таблицы:** vacancies, schedules


In [13]:
# Вопрос 2: q("""SELECT ...""")
q("""
SELECT extract(year FROM v.published_at)::int As year,
s.name AS schedule,
count(*) AS vacantion
FROM vacancies v                                                                                
JOIN schedules s ON s.schedule_id = v.schedule_id                                               
GROUP BY year, s.name
ORDER BY year, vacantion DESC;
""")

,year,schedule,vacantion
0,2019,Полный день,439485
1,2019,Удаленная работа,40400
2,2019,Гибкий график,35124
3,2019,Сменный график,14705
4,2019,Вахтовый метод,6242
5,2020,Полный день,450230
6,2020,Удаленная работа,78151
7,2020,Гибкий график,38095
8,2020,Сменный график,13723
9,2020,Вахтовый метод,7438


**Вывод:** 
Мы видим что явного перекоса в сторону какого та графика нет, от года к году наблюдается равномерный прирост числа вакансий в во всех графиках, за исключением сменного графика, кол0во вакансий с ним уменшилось.
Так же значительный перекос наблюдается на удаленный график, это может быть связанно с ковидной ситуацией сложившейся в 2020 году мы видим почти двукратный рост вакансий с удаленным графиком работы.


### 3. Какие валюты встречаются в вакансиях? Для каждой найдите число вакансий и медиану зарплаты. Почему эти медианы нельзя сравнивать между собой?

- **Таблицы:** vacancies


In [32]:
# Вопрос 3: q("""SELECT ...""")
q(
"""
SELECT c.currency_code, count(*)  AS vacancies,
percentile_cont(0.5) WITHIN GROUP (                                                      
             ORDER BY coalesce((salary_from + salary_to) / 2.0, salary_from, salary_to)           
         ) AS median_salary,                                                                      
         round(avg(coalesce((salary_from + salary_to) / 2.0, salary_from, salary_to))) AS         
  avg_salary 
FROM vacancies v
JOIN currencies c on c.currency_code = v.salary_currency
GROUP BY c.currency_code
ORDER BY count(*) DESC;
""")

,currency_code,vacancies,median_salary,avg_salary
0,RUR,494968,52500.0,69506.0
1,KZT,19571,180000.0,226377.0
2,USD,14486,1750.0,2223.0
3,BYR,12796,1000.0,1690.0
4,UAH,4585,16500.0,21745.0
5,EUR,3391,3000.0,3119.0
6,UZS,1717,3500000.0,4664919.0
7,KGS,282,27500.0,36683.0
8,AZN,114,850.0,1792.0
9,GEL,22,1000.0,2076.0


**Вывод:** 
Для сравнение выведена так же средняя зарплата рядом с медианной, для илюстрации "слона в комнате" видим что средняя стабильно больше но без эфекта Билла Гейтса на выборку. 
Медианны нельзя сравнить между собой так как они в разных валютах, при необходимости сравнени] между собой их неободимо перевести к одной единой валюте.

### 4. Какие медианы нижней и верхней границы рублёвой зарплаты у каждого уровня опыта? Выведите уровни в логическом порядке, а не по алфавиту.

- **Таблицы:** vacancies, experience_levels


In [49]:
# Вопрос 4: q("""SELECT ...""")

q(
"""
SELECT el.name,
percentile_cont(0.5) WITHIN GROUP (ORDER BY salary_from) AS median_from_salary,
percentile_cont(0.5) WITHIN GROUP (ORDER BY salary_to) AS median_to_salary
FROM vacancies v
JOIN experience_levels el ON el.experience_id = v.experience_id
WHERE v.salary_currency = 'RUR'
GROUP BY el.name, el.sort_order
ORDER BY el.sort_order;
""")

,name,median_from_salary,median_to_salary
0,Нет опыта,30000.0,45000.0
1,От 1 года до 3 лет,45000.0,70000.0
2,От 3 до 6 лет,80000.0,120000.0
3,Более 6 лет,100000.0,150000.0


**Вывод:** 
Заработная плата между опытом в 3-6 лет и более мало отличается. В среднем верхняя граница отличается от нижней на 50%.

### 5. Сколько рублёвых вакансий с подозрительной зарплатой: ниже 10 000 или выше 1 000 000? Посмотрите на примеры и решите, ошибки это или реальные значения.

- **Таблицы:** vacancies


In [60]:
# Вопрос 5: q("""SELECT ...""")

q(
"""
WITH s AS (
SELECT coalesce((salary_from + salary_to) / 2.0 , salary_from, salary_to) as mid
FROM vacancies
WHERE salary_currency = 'RUR'
)
SELECT CASE 
WHEN mid < 10_000 THEN 'ниже 10 000'
WHEN mid > 1_000_000 THEN 'больше 1 000 000'
ELSE 'Норма'
END AS Category,
count(*) as vacancies
FROM s
GROUP BY Category
ORDER BY vacancies DESC;
"""
)

,category,vacancies
0,Норма,493869
1,ниже 10 000,1045
2,больше 1 000 000,54


**Вывод:** 
В большинстве вакансий в зарплата указана в границах нормы.

## Блок 2. Соединения таблиц

Темы: JOIN, LEFT JOIN, анти-соединения и размножение строк.

### 6. Какие 10 городов дают больше всего вакансий и какая в них медианная рублёвая зарплата? Почему у Киева медиана пустая?

- **Таблицы:** vacancies, areas

In [11]:
# Вопрос 6: q("""SELECT ...""")
q("""
SELECT a.name, count(*) as vacancies,
percentile_cont(0.5) WITHIN GROUP (ORDER BY coalesce((salary_from + salary_to) / 2.0, salary_from, salary_to)           
         ) FILTER (WHERE v.salary_currency = 'RUR') AS median_rur 
FROM vacancies v
JOIN areas a ON a.area_id = v.area_id
GROUP BY a.area_id
ORDER BY vacancies DESC
LIMIT 10;
""")

# q("""SELECT * FROM vacancies LIMIT 5""")

,name,vacancies,median_rur
0,Москва,348705,85000.0
1,Санкт-Петербург,119272,65000.0
2,Минск,50481,50000.0
3,Новосибирск,32365,53400.0
4,Екатеринбург,28113,50000.0
5,Нижний Новгород,24729,45000.0
6,Казань,22700,50000.0
7,Киев,21142,NaN
8,Алматы,20717,105000.0
9,Краснодар,20099,48000.0


**Вывод:** 
Москва лидирует с большим отрывом, за ней идет Санкт Петербург, по Киеву медианная зарплата пуста так как валюта там гривны, но в выборку по ваканмсиям она попала, фильтр отсеял значения по валюте и в выборку попал NULL. К значению по Алматы нужно отнестись с осторожностью, оно посчитано на небольшом кол-ве вакансий в рублях которые для Казахстана считается иностранной валютой.


### 7. Кто 10 крупнейших работодателей 2020 года по числу вакансий и какую долю вакансий они публикуют с зарплатой? Почему группировать нужно по `employer_id`, а не по названию?

- **Таблицы:** vacancies, employers

In [50]:
# Вопрос 7: q("""SELECT ...""")
q("""
SELECT 
e.employer_id, e.name, count(*) as vacancies,
round(100.0 * count(*) FILTER (WHERE v.salary_from IS NOT NULL                   
                                           OR v.salary_to   IS NOT NULL)                  
               / count(*), 1) AS pct_with_salary
FROM vacancies v
JOIN employers e ON e.employer_id = v.employer_id
WHERE v.published_at >= '2020-01-01' AND v.published_at < '2021-01-01'
GROUP BY e.employer_id, e.name
ORDER BY vacancies DESC
LIMIT 10;
""")

,employer_id,name,vacancies,pct_with_salary
0,1740,Яндекс,11160,49.3
1,3529,Сбербанк,9744,1.3
2,2748,Ростелеком,9329,68.2
3,4934,Билайн,7866,72.9
4,3776,МТС,5563,17.1
5,1122462,Skyeng,4332,43.2
6,44272,ЭР-Телеком,3222,74.4
7,2630243,Ресурс Групп,2598,98.2
8,39305,Газпром нефть,2378,1.6
9,3127,МегаФон,2316,16.4


**Вывод:** 
Можем наблюдать что тройка крупнейших работодателей разместили больше вакансий чем остальные члены десятки. В среднем нормой является согласовывать заработную плату индивидуально, особенно для сбербанка и газпром нефть где подавляющие большинство вакансий без заработной платы. В отличии от Ресурс Групп где лишь для некоторых вакансий отсутсвует заработная плата. 
Групировать по employer_id надежнее на случай если имена работадателей совпадает. 

### 8. Сколько вакансий опубликовано без работодателя и чем они отличаются от остальных по доле вакансий с зарплатой и по медиане?

- **Таблицы:** vacancies, employers



In [36]:
# Вопрос 8: q("""SELECT ...""")
q("""
SELECT  
e.employer_id IS NULL AS no_employer, 
count(*) as vacancies,
round(100 * count(*) FILTER (WHERE v.salary_from IS NOT NULL OR v.salary_to   IS NOT NULL) / count(*), 2) AS pct_with_salary, 
percentile_cont(0.5) WITHIN GROUP
    (ORDER BY coalesce((v.salary_from + v.salary_to) / 2.0, v.salary_from, v.salary_to))
    FILTER (WHERE v.salary_currency = 'RUR') AS median_rur
FROM vacancies v
LEFT JOIN employers e ON e.employer_id = v.employer_id
GROUP BY no_employer                                                                    
ORDER BY no_employer;  
""")

,no_employer,vacancies,pct_with_salary,median_rur
0,False,1121380,49.0,52500.0
1,True,2213,49.0,100000.0


**Вывод:** 
Мы Видим что у вакансий без рабатодателя медианная заработная плата почти вдвое меньше тех где указан работадатель.
Вакансий без работадателя около 0.2% , можем сказать что подавляющее большинсво вакансий имеет работодателя. 

### 9. Какие работодатели публиковали вакансии в 2019 году, но не публиковали в 2020? Решите двумя способами и сравните планы через EXPLAIN. Кто из них был крупнейшим в 2019?

- **Таблицы:** vacancies, employers


In [47]:
# Вопрос 9: q("""SELECT ...""")
# Решение через EXCEPT
# q("""
# SELECT COUNT(*) FROM (
# SELECT employer_id FROM vacancies 
# WHERE published_at >= '2019-01-01' AND published_at < '2020-01-01' AND employer_id IS NOT NULL
# EXCEPT
# SELECT employer_id FROM vacancies 
# WHERE published_at >= '2020-01-01' AND published_at < '2021-01-01'
# ) t;
# """)

# Решение через NOT EXISTS
# q("""
# SELECT count(DISTINCT v.employer_id)                                                    
#   FROM vacancies v                                                                        
#   WHERE published_at >= '2019-01-01' AND published_at < '2020-01-01'                                                              
#     AND v.employer_id IS NOT NULL                                                         
#     AND NOT EXISTS (                                                                      
#         SELECT 1                                                                          
#         FROM vacancies v2                                                                 
#         WHERE v2.employer_id = v.employer_id                                                  
#           AND published_at >= '2020-01-01' AND published_at < '2021-01-01'                                                      
#     )
# """)
# общее решение
q(
"""
SELECT e.employer_id,                                                                   
         e.name,                                                                          
         count(*) AS vacancies_2019                                                       
  FROM vacancies v                                                                        
  JOIN employers e ON e.employer_id = v.employer_id                                       
  WHERE v.published_at >= '2019-01-01' AND v.published_at < '2020-01-01'                  
    AND NOT EXISTS (                                                                      
        SELECT 1                                                                          
        FROM vacancies v2                                                                 
        WHERE v2.employer_id = v.employer_id                                              
          AND v2.published_at >= '2020-01-01' AND v2.published_at < '2021-01-01'          
    )                                                                                     
  GROUP BY e.employer_id, e.name                                                          
  ORDER BY vacancies_2019 DESC                                                            
  LIMIT 10;
"""
)


,employer_id,name,vacancies_2019
0,2060911,ЮК-Консалтинг,515
1,1482197,Crossover,405
2,2297436,Mozgami,375
3,550468,Promotion Group,322
4,3456090,I Team,225
5,1639445,ПКП г. Челябинск,224
6,3787128,EOtradex,166
7,2741679,"Водоканал, г Рыбинск",138
8,2653802,ПиццаФабрика (ООО Всё и сразу),131
9,3159713,Akim agency,125


**Вывод:** 
Крупнейший работодатель ЮК-Консалтинг с 515 вакансиями. Если сравнить с результатами из 7 ячейки с сравнить с крупнейшими то даже самый крупный ЮК-Консалтинг не попадет и в топ 40 по общему кол-ву.

### 10. Сколько вакансий приходится на каждую отрасль? Почему сумма по отраслям больше общего числа вакансий и сколько вакансий вообще не попало ни в одну отрасль?

- **Таблицы:** vacancies, employer_industries, industries


In [15]:
# Вопрос 10: q("""SELECT ...""")

# main logic
q("""
SELECT i.industry_id, i.name, count(*) AS count_vacantion
FROM employer_industries ei
JOIN industries i ON ei.industry_id = i.industry_id
JOIN vacancies v ON ei.employer_id = v.employer_id
GROUP BY i.industry_id, i.name
ORDER BY count_vacantion DESC;
""")

# how many vacancion not include in any category
# q("""
# SELECT COUNT(*)
# FROM vacancies v
# LEFT JOIN employer_industries ei ON ei.employer_id = v.employer_id
# WHERE ei.employer_id IS NULL;
# """)

# q("""SELECT * FROM vacancies LIMIT 3""")

,industry_id,name,count_vacantion
0,39,Разработка программного обеспечения,287189
1,103,"Системная интеграция, автоматизации технологических и бизнес-процессов пред...",265266
2,232,"Интернет-компания (поисковики, платежные системы, соц.сети, информационно-по...",144974
3,5,Мобильная связь,76929
4,248,Банк,57363
...,...,...,...
292,261,"Лифтовое хозяйство (монтаж, сервис, ремонт)",79
293,156,Ритуальные услуги,24
294,8,"Ботанический сад, зоопарк, заповедник",22
295,126,Религиозная организация,5


**Вывод:** 
Сумма по отраслям больше чем кол-во вакансий так как на одну вакансию может приходится несколько отраслей это нормально.
199 254 вакансии без отрасли.

## Блок 3. Навыки: связь многие-ко-многим

Темы: связующая таблица, очистка строк, самосоединение.

### 11. Какие 20 навыков чаще всего указывали в 2020 году? Какую долю они составляют среди вакансий, где указан хотя бы один навык? Регистр пока не приводите.

- **Таблицы:** vacancy_skills, skills, vacancies


In [29]:
# Вопрос 11: q("""SELECT ...""")


# q("""
# SELECT s.name, count(*) AS vacancies
# FROM vacancy_skills vs
# JOIN skills s ON s.skill_id = vs.skill_id
# JOIN vacancies v ON vs.vacancy_id = v.vacancy_id
# WHERE v.published_at >= '2020-01-01' AND v.published_at < '2021-01-01'
# GROUP BY s.skill_id, s.name
# ORDER BY vacancies DESC
# LIMIT 20;
# """)

# second part about full vacation
q("""
SELECT count(DISTINCT vs.vacancy_id)                                                    
  FROM vacancy_skills vs                                                                  
  JOIN vacancies v ON vs.vacancy_id = v.vacancy_id                                                                 
  WHERE v.published_at >= '2020-01-01' AND v.published_at < '2021-01-01';    
""")

# q("""SELECT *  FROM vacancy_skills LIMIT 5;""")

,count
0,431738


**Вывод:** 
431 738 навыков, SQL 54 686 вакансий, (12,7 %) чаще всего встречались в 2020 году

### 12. Сколько разных написаний у одних и тех же навыков? У каких навыков их больше всего и сколько записей в `skills` останется, если не учитывать регистр?

- **Таблицы:** skills

In [67]:
# Вопрос 12: q("""SELECT ...""")

# first part how many full and unique
# q("""
# SELECT count(*)           AS total,                                                         
#        count(DISTINCT lower(name))  AS unique_lower                                                   
#   FROM skills;
# """)

# filter by type compination
# q("""
# SELECT lower(name) AS skill_key,                                                        
#          count(*)    AS variants                                                          
#   FROM skills                                                                             
#   GROUP BY lower(name)
#   ORDER BY variants DESC; 
# """)

# how many comb > 1 / 7687
# q("""
# SELECT COUNT(*) FROM (
# SELECT lower(name) AS skill_key,                                                        
#          count(*)    AS variants                                                          
#   FROM skills                                                                             
#   GROUP BY lower(name)
#   HAVING count(*) > 1    
# ) t;
# """)

q("""
SELECT lower(name) AS skill_key,                                                        
         count(*)    AS variants,
         array_agg(name ORDER BY name) AS array_words
  FROM skills                                                                             
  GROUP BY lower(name)
  HAVING count(*) > 1
  ORDER BY variants DESC, skill_key
  LIMIT 10;
""")


# q("""SELECT * FROM skills""")

,skill_key,variants,array_words
0,asp.net core,10,"[asp.net core, Asp.net core, Asp.Net Core, Asp.NET Core, ASP.net core, ASP.N..."
1,restful api,10,"[restful api, restful API, Restful api, Restful Api, Restful API, RestFul Ap..."
2,.net core,8,"[.net core, .net Core, .Net core, .Net Core, .Net CORE, .NET core, .NET Core..."
3,devops,8,"[devops, devOps, Devops, DevOps, DevOpS, DevOPs, DevOPS, DEVOPS]"
4,mssql,8,"[mssql, Mssql, MsSql, MsSQL, MSsql, MSSql, MSSQl, MSSQL]"
5,rest api,8,"[rest api, rest API, Rest api, Rest Api, Rest API, REST api, REST Api, REST ..."
6,web api,8,"[web api, web Api, web API, Web api, Web Api, Web API, WEB Api, WEB API]"
7,webapi,8,"[webapi, webApi, webAPI, Webapi, WebApi, WebAPI, WEBapi, WEBAPI]"
8,cms opencart,7,"[cms opencart, cms OpenCart, Cms opencart, CMS opencart, CMS Opencart, CMS O..."
9,gitlab,7,"[gitlab, Gitlab, GitLab, GitLAB, GItLab, GITlab, GITLAB]"


**Вывод:** 
Предусмортели узкое место с самым богатым вариантом, так как кандидата 2 добавили второй аргумент в 
   ORDER BY для гарантии одинаковых выводов при многократных запусках.

### 13. Какие 10 навыков чаще всего указаны в одной вакансии с Python?

- **Таблицы:** vacancy_skills, skills
- **Что потренировать:** CTE или самосоединение vacancy_skills, lower()
- **Контроль:** вакансий с Python: 41 637; на первом месте linux: 14 375.

In [6]:
# Вопрос 13: q("""SELECT ...""")
# CTE -- find vacancies with Python
# Main part concat cte with self and check other skills on vac how have targer
q("""
WITH py AS (
SELECT DISTINCT vs.vacancy_id
FROM vacancy_skills vs
JOIN skills s ON vs.skill_id = s.skill_id
WHERE lower(s.name) = 'python'
)
SELECT lower(s.name) AS skill, 
       count(DISTINCT vs.vacancy_id) as vacancies
FROM py
JOIN vacancy_skills vs ON py.vacancy_id = vs.vacancy_id
JOIN skills s          ON  s.skill_id = vs.skill_id
WHERE lower(s.name) <> 'python'
GROUP BY lower(s.name)
ORDER BY vacancies DESC, skill                                                          
LIMIT 10;                      
""")



,skill,vacancies
0,linux,14375
1,sql,13768
2,git,10529
3,postgresql,8347
4,java,6466
5,javascript,5820
6,django framework,5414
7,c++,5097
8,bash,4941
9,mysql,4122


**Вывод:** 

### 14. Сколько навыков в среднем указано в вакансии на каждом уровне опыта? Посчитайте двумя способами: среди всех вакансий и только среди вакансий с навыками. Какая доля вакансий без навыков?

- **Таблицы:** vacancies, vacancy_skills, experience_levels
- **Что потренировать:** LEFT JOIN, COUNT(column) против COUNT(*), CTE
- **Контроль:** 4 строки; «Нет опыта»: 3,24 среди всех, 5,84 среди вакансий с навыками, 44,5 % без навыков.

In [10]:
# Вопрос 14: q("""SELECT ...""")
q("""                                                                                   
  WITH per_vac AS (                                                                       
      SELECT v.vacancy_id,                                                                
             v.experience_id,                                                             
             count(vs.skill_id) AS n_skills                                               
      FROM vacancies v                                                                    
      LEFT JOIN vacancy_skills vs ON vs.vacancy_id = v.vacancy_id                         
      GROUP BY v.vacancy_id, v.experience_id                                              
  )                                                                                       
  SELECT el.name,                                                                         
         count(*)                                              AS vacancies,              
         round(avg(n_skills), 2)                               AS avg_all,                
         round(avg(n_skills) FILTER (WHERE n_skills > 0), 2)   AS avg_with_skills,        
         round(100.0 * count(*) FILTER (WHERE n_skills = 0)                               
               / count(*), 1)                                  AS pct_no_skills           
  FROM per_vac p                                                                          
  JOIN experience_levels el ON el.experience_id = p.experience_id                         
  GROUP BY el.experience_id, el.name, el.sort_order                                       
  ORDER BY el.sort_order;                                                                 
  """)           

,name,vacancies,avg_all,avg_with_skills,pct_no_skills
0,Нет опыта,207772,3.24,5.84,44.5
1,От 1 года до 3 лет,577373,3.91,6.34,38.3
2,От 3 до 6 лет,307125,3.95,6.26,36.9
3,Более 6 лет,31323,4.43,7.07,37.4


**Вывод:** 
Чем больше опыта требуется в вакансии, тем больше навыков в ней указано. В вакансиях без опыта у половины вакансий отсутсвует опыт вовсе, и в целом выдим что для трети рынка вакансий характерно не иметь вакансий. Тоесть ислеование в вопросах 11-13 из проекта иследуют только две трети рынка. 

### 15. Разделите вакансии на три группы: только SQL, только Python, оба навыка. Сколько вакансий в каждой группе и какая медианная рублёвая зарплата? Приведите названия навыков к нижнему регистру.

- **Таблицы:** vacancy_skills, skills, vacancies
- **Что потренировать:** bool_or, условная агрегация, CASE
- **Контроль:** 3 строки; оба навыка: 13 768 вакансий, 100 000; только Python: 27 869, 105 000; только SQL: 66 547, 85 000.

In [14]:
# Вопрос 15: q("""SELECT ...""")
q("""
 WITH flags AS (                                                                         
      SELECT vs.vacancy_id,                                                               
             bool_or(lower(s.name) = 'python') AS has_python,                             
             bool_or(lower(s.name) = 'sql')    AS has_sql                                 
      FROM vacancy_skills vs                                                              
      JOIN skills s ON s.skill_id = vs.skill_id                                           
      WHERE lower(s.name) IN ('python', 'sql')                                            
      GROUP BY vs.vacancy_id                                                              
  ),
   groups AS (                                                                             
      SELECT vacancy_id,                                                                  
             CASE                                                                         
                 WHEN has_python AND has_sql THEN 'оба навыка'                            
                 WHEN has_python             THEN 'только Python'                         
                 ELSE                             'только SQL'                            
             END AS grp                                                                   
      FROM flags                                                                          
  ) 
  SELECT g.grp,                                                                           
         count(*) AS vacancies,                                                           
         percentile_cont(0.5) WITHIN GROUP (                                              
             ORDER BY coalesce((v.salary_from + v.salary_to) / 2.0, v.salary_from,        
  v.salary_to)                                                                            
         ) FILTER (WHERE v.salary_currency = 'RUR') AS median_rur                         
  FROM groups g                                                                           
  JOIN vacancies v ON v.vacancy_id = g.vacancy_id                                         
  GROUP BY g.grp                                                                          
  ORDER BY g.grp; 
""") 

,grp,vacancies,median_rur
0,оба навыка,13768,100000.0
1,только Python,27869,105000.0
2,только SQL,66547,85000.0


**Вывод:**  SQL - очень массовый навык, охватывает большое кол-во вакансий, многим он нужен, и поэтому он уступает где требуется только
Python, это могут быть вакансии разработчиков и ML специалистов где навык SQL может быть не вынесен, но подразумеваться. 
Оба навыка уступают по ко-во обоим по отдельности, логика аналогично предыдущим, но еще накладывается вариативность фильтров, чем больше фильтров тем меньше возможных уникальных значений. (Из 3 значений комбинация - единтсвенный где переменных 2 остальные варианты свободнее)

## Блок 4. Оконные функции

Темы: LAG, скользящие окна, ранжирование, NTILE.

### 16. Как менялось число вакансий от месяца к месяцу, в процентах? В какие месяцы были самые сильные падения и похоже ли это на изменения рынка или на особенность сбора данных?

- **Таблицы:** vacancies
- **Что потренировать:** date_trunc, LAG
- **Контроль:** 24 строки; самые сильные падения: ноябрь 2020 (−43,7 %) и апрель 2020 (−35,4 %).

In [32]:
# Вопрос 16: q("""SELECT ...""")
q("""
WITH monthly AS (                                                                       
      SELECT date_trunc('month', published_at)::date AS month,
      count(*) AS vacanc
      FROM vacancies
      GROUP BY month
      ORDER BY month )

SELECT   month,                                                                           
         vacanc,                                                                             
         LAG(vacanc) OVER (ORDER BY month) AS prev_vacanc,                                          
         round(100  * (vacanc - LAG(vacanc) OVER (ORDER BY month))                                
            / LAG(vacanc) OVER (ORDER BY month) , 1 ) AS pct_change                                                 
FROM monthly                                                                            
ORDER BY month; 
""")
# q("""SELECT * FROM vacancies LIMIT 3""")

,month,vacanc,prev_vacanc,pct_change
0,2019-01-01,39807,NaN,NaN
1,2019-02-01,43450,39807.0,9.0
2,2019-03-01,43916,43450.0,1.0
3,2019-04-01,45627,43916.0,3.0
4,2019-05-01,42711,45627.0,-6.0
5,2019-06-01,42110,42711.0,-1.0
6,2019-07-01,47538,42110.0,12.0
7,2019-08-01,45791,47538.0,-3.0
8,2019-09-01,46652,45791.0,1.0
9,2019-10-01,51211,46652.0,9.0


**Вывод:** 
Мы видим значительное падение на 35% в апреле 2020 это можем связать с кодидом.         
  И рост, в июне того же года, В москве отменили режим самоизоляции, сказался эфект низкой базы, на рынок вернулись вакансии которого его лишили в апреле. 
  В сентябре снова можем видеть рост на 30% -это можем связать с частичным снятием        
  ограничений. А в ноябре падание на 43% стоит иследовать, похоже на возврат к норме после анамального октября, в таком случае можем это связать с особеностью сбора данных, и возложить на авторов датасета. 

### 17. Постройте скользящее среднее числа вакансий за 7 дней. Есть ли дни без единой вакансии, и как они искажают окно `ROWS BETWEEN 6 PRECEDING`?

- **Таблицы:** vacancies
- **Что потренировать:** generate_series, LEFT JOIN, AVG() OVER (ROWS BETWEEN ...)
- **Контроль:** 731 день в календаре, из них 4 дня подряд в октябре 2020 без вакансий; пик скользящего среднего: 15 октября 2020, около 4 528.

In [43]:
# Вопрос 17: q("""SELECT ...""")

q("""                                                                                   
  WITH calendar AS (                                  
      SELECT d::date AS day                                                               
      FROM generate_series('2019-01-01'::date, '2020-12-31'::date, interval '1 day') AS d 
  ),                                                                                      
  daily AS (                               
      SELECT published_at::date AS day,                                                   
             count(*) AS vacanc                                                           
      FROM vacancies                                                                      
      GROUP BY day                                                                        
  ),                                                                                      
  full_series AS (                           
      SELECT c.day,                                                                       
             coalesce(d.vacanc, 0) AS vacancies,                                          
             round(avg(coalesce(d.vacanc, 0)) OVER (                                      
                   ORDER BY c.day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 1) AS ma_full
      FROM calendar c                                                                     
      LEFT JOIN daily d ON d.day = c.day                                                  
  ),                                                                                      
  gaps AS (                                                                                                                      
      SELECT day,                                                                         
             round(avg(vacanc) OVER (                                                     
                   ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 1) AS ma_gaps  
      FROM daily                                                                          
  )                                                                                       
  SELECT f.day,                                                                           
         f.vacancies,                                                                     
         f.ma_full,                                                                       
         g.ma_gaps                                                                        
  FROM full_series f                                                                      
  LEFT JOIN gaps g ON g.day = f.day               
  WHERE f.day BETWEEN '2020-10-14' AND '2020-10-30'                                       
  ORDER BY f.day;                                                                         
  """)            

,day,vacancies,ma_full,ma_gaps
0,2020-10-14,6179,4386.3,4386.3
1,2020-10-15,5159,4528.0,4528.0
2,2020-10-16,3932,4510.6,4510.6
3,2020-10-17,197,4414.4,4414.4
4,2020-10-18,7,4164.6,4164.6
5,2020-10-19,0,3086.0,NaN
6,2020-10-20,0,2210.6,NaN
7,2020-10-21,0,1327.9,NaN
8,2020-10-22,0,590.9,NaN
9,2020-10-23,872,153.7,3210.6


**Вывод:** 
Календарь за 2019–2020 годы содержит 731 день, а вакансии опубликованы только в 727 из  
  них. Четыре дня подряд, 19–22 октября 2020, без единой вакансии. Провал на самом деле   
  шире: 17–18 и 23–25 октября тоже почти пустые (от 7 до 872 вакансий при обычных 4–6     
  тысячах в будни). Скорее всего, это сбой сбора данных, а не событие на рынке: рынок не  
  останавливается на неделю, и обрыв ничем не объясняется. 

### 18. Какие 5 навыков самые частые в каждом из классов `programmer`, `analyst`, `tester`, `sysadmin`, `designer`?

- **Таблицы:** vacancy_prof_classes, vacancy_skills, skills
- **Что потренировать:** ROW_NUMBER() OVER (PARTITION BY ...), CTE
- **Контроль:** 25 строк; analyst на первом месте: SQL (14 589); designer: Adobe Photoshop (9 250).

In [55]:
# Вопрос 18: q("""SELECT ...""")

q("""
 WITH counts AS (                                                                        
      SELECT vp.prof_class,                                                                   
         s.name,                                                                          
         count(*) AS vacancies                                                            
  FROM vacancy_skills vs                                                                  
  JOIN vacancy_prof_classes vp ON vp.vacancy_id = vs.vacancy_id                           
  JOIN skills s                ON s.skill_id    = vs.skill_id                             
  WHERE vp.prof_class IN ('programmer', 'analyst', 'tester', 'sysadmin', 'designer')      
  GROUP BY vp.prof_class, s.skill_id, s.name                                                      
  ),                                                                                      
  ranked AS (                                                                             
      SELECT *,                                                                           
              ROW_NUMBER() OVER (PARTITION BY prof_class ORDER BY vacancies DESC, name) AS rn                      
      FROM counts                                                                         
  )                                                                                       
  SELECT *                                                                                
  FROM ranked                                                                             
  WHERE rn <= 5                                                                               
  ORDER BY prof_class, rn; 
""")

,prof_class,name,vacancies,rn
0,analyst,SQL,14589,1
1,analyst,Бизнес-анализ,9294,2
2,analyst,UML,6066,3
3,analyst,Разработка технических заданий,6039,4
4,analyst,MS SQL,5742,5
5,designer,Adobe Photoshop,9250,1
6,designer,Adobe Illustrator,5228,2
7,designer,UX,4177,3
8,designer,UI,3838,4
9,designer,Web-дизайн,3553,5


**Вывод:** 
 - SQL — сквозной навык. Он первый у аналитиков и тестировщиков и третий у программистов.
    Это самый универсальный навык в IT-вакансиях, что согласуется с задачами 11 и 15.     
  - Профили классов хорошо различимы. У программистов веб-стек (JavaScript, CSS, HTML) и  
    Git. У аналитиков — бизнес-анализ, UML и ТЗ. У сисадминов — Linux, Windows и сети. У  
    дизайнеров — инструменты Adobe и UX/UI.                                               
  - У тестировщиков SQL опережает «Тестирование» всего на одну вакансию. Без второго ключа
    сортировки их порядок был бы случайным.                                               
  - Видны дубли из задачи 12: у аналитиков отдельно стоят SQL и MS SQL, у сисадминов —    
    устаревший «Windows 7». Если привести названия к единому виду, топ мог бы немного     
    измениться.                                                                           
                          

### 19. Какие навыки сильнее всего поднялись в рейтинге с 2019 на 2020 год? Учитывайте только навыки, которые встретились минимум в 500 вакансиях в каждом из лет, и не различайте регистр. О чём говорит этот список?

- **Таблицы:** vacancy_skills, skills, vacancies
- **Что потренировать:** RANK(), две CTE, соединение таблицы с собой
- **Контроль:** первое место: «работа с людьми», с 436-го места на 203-е.

In [67]:
# Вопрос 19: q("""SELECT ...""")

q("""
 WITH counts AS (                                                                        
      SELECT extract(year FROM v.published_at)::int AS yr,                                
             lower(s.name)                          AS skill,                             
             count(*)                               AS vacancies                          
      FROM vacancy_skills vs                                                              
      JOIN skills s    ON s.skill_id   = vs.skill_id                                      
      JOIN vacancies v ON v.vacancy_id = vs.vacancy_id                                    
      GROUP BY yr, skill                                                                  
  ),                                                                                       
   ranked AS (SELECT yr, skill, vacancies,                                                            
         RANK() OVER (PARTITION BY yr ORDER BY vacancies DESC) AS rnk                     
  FROM counts) 

  SELECT 
  a.skill,                                         
         a.vacancies          AS vacancies_2019,                                          
         b.vacancies          AS vacancies_2020,                                          
         a.rnk                AS rank_2019,                                               
         b.rnk                AS rank_2020,                                               
         a.rnk - b.rnk        AS rank_up     
  FROM ranked a
  JOIN ranked b ON a.skill = b.skill
  WHERE a.yr = 2019                                                                       
    AND b.yr = 2020                                                                       
    AND a.vacancies >= 500                                                                
    AND b.vacancies >= 500                                                                
  ORDER BY rank_up DESC, a.skill                                                          
  LIMIT 15; 
  
""")

# q("""SELECT * FROM vacancy_skills LIMIT 10""")

,skill,vacancies_2019,vacancies_2020,rank_2019,rank_2020,rank_up
0,работа с людьми,548,2743,436,203,233
1,прием и распределение телефонных звонков,591,2930,418,191,227
2,управление рисками,514,2468,452,225,227
3,обучение,505,2197,456,240,216
4,управление временем,856,3728,328,153,175
5,планирование,631,2575,389,216,173
6,навыки межличностного общения,732,3039,353,185,168
7,аналитика,619,2102,402,247,155
8,ориентация на клиента,557,1781,433,280,153
9,разработка инструкций,621,2087,399,250,149


**Вывод:** 
"работа с людьми" по Среди лидеров роста почти нет конкретных технологий. Это общие    
  «мягкие» и офисные навыки, и число вакансий с ними выросло в 3–5 раз за год. Вряд ли    
  рынок за один год так резко изменил требования. Скорее изменилось то, как работодатели  
  заполняют поле навыковднялась на 233 мест примерно так же поднялось "Прием и распределение телефонных звонков" и "управление рисками" 


### 20. Разделите работодателей минимум с 50 рублёвыми вакансиями с зарплатой на 4 группы по медианной зарплате. Какие границы у каждой группы и кто входит в верхнюю?

- **Таблицы:** vacancies, employers
- **Что потренировать:** HAVING, NTILE(4)
- **Контроль:** 1 255 работодателей; верхняя группа начинается от 100 000; первое место: Lafa games, 550 000.

In [76]:
# Вопрос 20: q("""SELECT ...""")

q(""" 
WITH emp AS (
SELECT e.employer_id,                                                                   
         e.name,                                                                          
         count(*) AS vacancies,                                                           
         percentile_cont(0.5)
         WITHIN GROUP (ORDER BY coalesce((v.salary_from + v.salary_to) / 2.0, v.salary_from, v.salary_to)) AS median_salary   
  FROM vacancies v                                                                        
  JOIN employers e ON e.employer_id  = v.employer_id                                                           
  WHERE v.salary_currency = 'RUR'                                                         
    AND (v.salary_from IS NOT NULL OR v.salary_to IS NOT NULL)                                                                             
  GROUP BY e.employer_id, e.name                                                                           
  HAVING count(*) >= 50 
),
q AS(
SELECT *,                                                                               
         NTILE(4) OVER (ORDER BY median_salary) AS quartile                                                  
  FROM emp 
   ORDER BY median_salary DESC
) 

SELECT quartile,                                                                        
         count(*),                                                           
         min(median_salary) AS lower_bound,                                                                
         max(median_salary) AS upper_bound                                                                
  FROM q                                                                                  
  GROUP BY quartile                                                                            
  ORDER BY quartile; 
""")

# q("""
# SELECT * FROM vacancies LIMIT 3;
# """)

,quartile,count,lower_bound,upper_bound
0,1,314,5380.0,45000.0
1,2,314,45000.0,65000.0
2,3,314,65000.0,100000.0
3,4,313,100000.0,550000.0


**Вывод:**
 Работодатели с 50+ рублёвыми вакансиями с зарплатой (1 255 компаний) разделены на четыре равные группы по медианной зарплате: до 45 тыс., 45–65 тыс., 65–100 тыс. и от 100 тыс. до 550 тыс. Верхняя группа самая «растянутая»: её разброс (450 тыс.) больше, чем у трёх остальных вместе. Зарплатное распределение сильно скошено вправо. В верхней группе в основном IT-компании и игровые студии (Lafa games, DriverPack Solution, СофтПро).

## Блок 5. Производительность, представления и качество данных

Темы: EXPLAIN, VIEW, регулярные выражения, итоговый отчёт.

### 21. Сравните планы двух запросов, считающих вакансии за март 2020: с условием `published_at >= '2020-03-01' AND published_at < '2020-04-01'` и с условием через `extract(year ...)` и `extract(month ...)`. Почему второй медленнее, хотя оба используют один индекс?

- **Таблицы:** vacancies
- **Что потренировать:** EXPLAIN ANALYZE, индексы, условия с функцией над колонкой
- **Контроль:** оба дают 53 866; первый читает только нужный диапазон индекса, а второй просматривает его почти целиком и отбрасывает остальное (Rows Removed by Filter). Разница примерно в 20 раз.

In [91]:
# Вопрос 21: q("""SELECT ...""")
# q("""
# SELECT
#       count(*) AS vacanc
#       FROM vacancies
#       WHERE published_at >= '2020-03-01' AND published_at < '2020-04-01'
# """)

# q("""
#  SELECT count(*) AS vacanc                                                               
#   FROM vacancies                                                                          
#   WHERE extract(month FROM published_at) = 3                                              
#     AND extract(year FROM published_at) = 2020  
# """)


# Aggregate  (cost=1878.04..1878.05 rows=1 width=8) (actual time=21.531..21.532 rows=1 loops=1)
#   ->  Index Only Scan using ix_vacancies_published on vacancies  (cost=0.43..1742.75 rows=54116 width=0) (actual time=0.132..17.611 rows=53866 loops=1)
#         Index Cond: ((published_at >= '2020-03-01 00:00:00+03'::timestamp with time zone) AND (published_at < '2020-04-01 00:00:00+03'::timestamp with time zone))
#         Heap Fetches: 0
# Planning Time: 0.181 ms
# Execution Time: 21.598 ms

# plan_a = q("""                                                                          
#   EXPLAIN ANALYZE                                                                         
#   SELECT count(*) FROM vacancies                                                          
#   WHERE published_at >= '2020-03-01' AND published_at < '2020-04-01'                      
#   """)                                                                                    
# print("\n".join(plan_a["QUERY PLAN"]))  




# Aggregate  (cost=1878.04..1878.05 rows=1 width=8) (actual time=19.249..19.250 rows=1 loops=1)
#   ->  Index Only Scan using ix_vacancies_published on vacancies  (cost=0.43..1742.75 rows=54116 width=0) (actual time=4.459..15.887 rows=53866 loops=1)
#         Index Cond: ((published_at >= '2020-03-01 00:00:00+03'::timestamp with time zone) AND (published_at < '2020-04-01 00:00:00+03'::timestamp with time zone))
#         Heap Fetches: 0
# Planning Time: 0.300 ms
# Execution Time: 19.294 ms
plan_b = q("""                                                                          
  EXPLAIN ANALYZE                                                                         
  SELECT count(*) AS vacanc FROM vacancies                                                                          
  WHERE extract(month FROM published_at) = 3 AND extract(year FROM published_at) = 2020                     
  """)                                                                                    
print("\n".join(plan_b["QUERY PLAN"]))  

Finalize Aggregate  (cost=29618.42..29618.43 rows=1 width=8) (actual time=87.635..89.416 rows=1 loops=1)
  ->  Gather  (cost=29618.20..29618.41 rows=2 width=8) (actual time=87.537..89.412 rows=3 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        ->  Partial Aggregate  (cost=28618.20..28618.21 rows=1 width=8) (actual time=77.454..77.455 rows=1 loops=3)
              ->  Parallel Index Only Scan using ix_vacancies_published on vacancies  (cost=0.43..28618.17 rows=12 width=0) (actual time=46.734..76.873 rows=17955 loops=3)
                    Filter: ((EXTRACT(month FROM published_at) = '3'::numeric) AND (EXTRACT(year FROM published_at) = '2020'::numeric))
                    Rows Removed by Filter: 356576
                    Heap Fetches: 0
Planning Time: 0.151 ms
Execution Time: 89.447 ms


**Вывод:** 
Оба запроса возвращают 53 866 и используют один индекс ix_vacancies_published, но     
по-разному.                                                                           
                                                                                       
 Запрос с диапазоном использует условие как Index Cond: Postgres сразу находит в       
   индексе 1 марта и читает ровно 53 866 записей до 1 апреля, ничего лишнего. Время около
   20 мс.                                                                                
                                                                                         
   Запрос с extract использует условие как Filter. Индекс упорядочен по самим датам, а не
   по результату extract, поэтому найти «месяц = 3» сразу невозможно. Postgres           
   пролистывает индекс целиком (~1,12 млн записей, вся таблица), вычисляет extract для   
   каждой строки и выбрасывает 1,07 млн (Rows Removed by Filter). Оценка стоимости выше в
   16 раз, время — в 4,5 раза, несмотря на три параллельных процесса. Планировщик к тому 
   же ошибается в оценке числа строк (12 вместо ~18 тыс.), потому что не умеет           
   предсказывать результат функции.                                                      
                                                                                         
   Правило: не оборачивайте индексированный столбец функцией в WHERE. Условие должно     
   сравнивать «голый» столбец с константами, например полуинтервалом >= начало AND <     
   конец. Если функция всё же нужна, понадобится индекс по этому же выражению.

### 22. Создайте представление `v_salary`: вакансия, год, `mid` и чистая зарплата `net` (при `salary_gross = true` умножьте на 0,87), только рубли и только `mid` от 10 000 до 1 000 000. С его помощью найдите медиану `net` по годам и уровням опыта.

- **Таблицы:** vacancies, experience_levels
- **Что потренировать:** CREATE VIEW (выполняйте через `run()`), затем SELECT
- **Контроль:** 8 строк; 2020 год, «От 1 года до 3 лет»: 52 200.

In [96]:
# Вопрос 22: q("""SELECT ...""")

# run("""
#   CREATE VIEW v_salary AS 
#   SELECT vacancy_id,                                                                      
#          year,                                                                            
#          mid,                                                                             
#          CASE WHEN salary_gross THEN mid * 0.87                                           
#               ELSE mid                                                                    
#          END AS net                                                                       
#   FROM (                                                                                  
#       SELECT vacancy_id,                                                                  
#              extract(year FROM published_at)::int AS year,                                
#              coalesce((salary_from + salary_to) / 2.0, salary_from, salary_to) AS mid,    
#              salary_gross                                                                 
#       FROM vacancies                                                                      
#       WHERE salary_currency = 'RUR'                                                       
#   ) s                                                                                     
#   WHERE mid BETWEEN 10000 AND 1000000;                                                                               
# """)  

q("""
 SELECT s.year,                                                                          
         el.name,                                                                         
         percentile_cont(0.5) WITHIN GROUP (ORDER BY s.net) AS median_net                 
  FROM v_salary s                                                                         
  JOIN vacancies v          ON s.vacancy_id = v.vacancy_id                                                       
  JOIN experience_levels el ON el.experience_id = v.experience_id                                                        
  GROUP BY s.year, el.name, el.sort_order                                                 
  ORDER BY s.year, el.sort_order;  
""")

,count
0,493869


**Вывод:** 
  Создано представление v_salary: рублёвые вакансии с серединой вилки mid от 10 тыс. до 
   1 млн и зарплатой «на руки» net (для gross-вакансий mid × 0,87). Через него посчитаны 
   медианы net по годам и уровням опыта.                                                 
                                                                                         
   - Опыт — главный фактор зарплаты. В 2020 году медиана растёт от 31 тыс. (без опыта) до
   52 тыс. (1–3 года), 90 тыс. (3–6 лет) и 125 тыс. (более 6 лет). Самый большой скачок, 
   почти в 1,7 раза, — между «1–3 года» и «3–6 лет».                                     
   - С 2019 по 2020 год медианы почти не изменились: рост на 2–4 % в младших группах и 0 
   % у самых опытных (125 тыс. в оба года). С учётом инфляции (~3–5 %) реальные зарплаты 
   в 2020 году скорее стояли на месте. Это согласуется с ковидным годом.                 
   - Допущение: вакансии с неизвестным salary_gross (около 2,4 тыс.) считаются «на руки»,
   без пересчёта. 

### 23. В каких вакансиях 2020 года Docker упомянут в описании, но не указан в навыках? Разбейте вакансии на 4 группы по двум признакам: есть в навыках и есть в описании. Что это говорит о надёжности поля навыков?

- **Таблицы:** vacancies, vacancy_descriptions, vacancy_skills, skills
- **Что потренировать:** регулярное выражение `~*`, EXISTS. Строку Python пишите как `r"""..."""`, иначе `\m` и `\M` испортятся
- **Контроль:** 4 строки; только в описании: 30 092; и там, и там: 5 765. Условие: `description ~* '\mdocker\M'`, запрос идёт около 20 секунд.

In [104]:
# Вопрос 23: q("""SELECT ...""")

# q(r"""                                                                                  
#   SELECT txt,                                                                             
#          txt ~* 'docker'          AS simple,                                              
#          txt ~* '\mdocker\M'      AS word                                                 
#   FROM (VALUES                                                                            
#       ('Опыт работы с Docker'),                                                           
#       ('знание DOCKER и k8s'),                                                            
#       ('docker-compose'),                                                                 
#       ('используем dockerfile'),                                                          
#       ('Python, SQL')                                                                     
#   ) AS t(txt);                                                                            
#   """) 

q(r""" 

  WITH flags AS(
  SELECT v.vacancy_id,                                                                    
          coalesce(d.description ~* '\mdocker\M', false) AS in_desc,
          EXISTS (                                                                                
                  SELECT 1                                                                            
                  FROM vacancy_skills vs                                                              
                  JOIN skills s ON s.skill_id = vs.skill_id                                                              
                  WHERE vs.vacancy_id = v.vacancy_id                                                        
                  AND  lower(s.name) = 'docker'                                                               
                 ) AS in_skills
  FROM vacancies v                                                                        
  LEFT JOIN vacancy_descriptions d ON d.vacancy_id = v.vacancy_id                                                  
  WHERE v.published_at >= '2020-01-01' AND v.published_at < '2021-01-01'
  
  )
  SELECT in_skills,                                                                       
         in_desc,                                                                         
         count(*) AS vacancies                                                            
  FROM flags                                                                              
  GROUP BY in_skills, in_desc                                                             
  ORDER BY in_skills, in_desc;   
  """)






,in_skills,in_desc,vacancies
0,False,False,551046
1,False,True,30092
2,True,False,734
3,True,True,5765


**Вывод:** 
Docker упомянут в 36,6 тыс. вакансиях 2020 года, но в поле навыков указан только в 6,5
тыс. (18 %). В 30 092 вакансиях (82 %) он есть только в описании. Обратная ситуация   
(только в навыках, 734) встречается редко. Скорее всего, это вакансии, где описание   
короткое или написано общими словами.                                                 
                                                                                       
 Поле навыков ненадёжно как единственный источник. Работодатели заполняют его          
 выборочно, а реальные требования пишут в тексте. Поэтому рейтинги навыков из задач    
 11–19 занижают популярность технических инструментов, а «мягкие» навыки, которые      
 удобно выбирать из подсказок hh.ru, получают в них преимущество (вспомните задачу 19).
 Для точной оценки спроса на технологию нужно искать её и в описаниях. Поиск по слову  
 с границами \m...\M защищает от ложных совпадений вроде dockerfile.                   
                                                                                         
   Цена такой точности — скорость: поиск по тексту занимает около 20 секунд, потому что  
   регулярное выражение проверяется на каждом описании, и обычный индекс тут не помогает 
   (урок задачи 21: функция над столбцом).     

### 24. Совпадает ли `address_city` с регионом вакансии в `areas`? Сколько вакансий без адреса, сколько с совпадением, а сколько с расхождением? Какие из расхождений настоящие ошибки, а какие просто другое написание?

- **Таблицы:** vacancies, areas
- **Что потренировать:** сравнение строк, COUNT с FILTER, разбор примеров
- **Контроль:** заполнено 542 616; совпало без учёта регистра 499 487; расхождений 43 129; частый пример: Нур-Султан и Астана.

In [118]:
# Вопрос 24: q("""SELECT ...""")

# main part   
# q("""
# SELECT count(*)                                                      AS total,                             
#          count(v.address_city)                                       AS filled,
#          count(*) FILTER (WHERE v.address_city IS NULL)                        AS no_address,
#          count(*) FILTER (WHERE lower(v.address_city) = lower(a.name)) AS matched,
#          count(*) FILTER (WHERE lower(v.address_city) != lower(a.name))                AS mismatched 
                              
#   FROM vacancies v                                                                        
#   JOIN areas a ON a.area_id = v.area_id; 
# """)


q("""
 SELECT a.name          AS area,                                                         
         v.address_city,                                                                  
         count(*)        AS vacancies                                                     
  FROM vacancies v                                                                        
  JOIN areas a ON a.area_id = v.area_id                                                   
  WHERE lower(v.address_city) != lower(a.name)                                                                               
  GROUP BY a.name, v.address_city                                                                         
  ORDER BY vacancies DESC                                                                         
  LIMIT 20;       
""")

,area,address_city,vacancies
0,Нур-Султан,Астана,2241
1,Киров (Кировская область),Киров,2151
2,Нур-Султан,Нур-Султан (Астана),1552
3,Иваново (Ивановская область),Иваново,1354
4,Москва,Новомосковский административный округ,1163
5,Орел,Орёл,761
6,Москва,деревня Румянцево,621
7,Москва,Химки,458
8,Могилев,Могилёв,438
9,Благовещенск (Амурская область),Благовещенск,436


**Вывод:** 
Мы Можем заключить что первые 5 строк отданы разному написанию одного и того же города, тут можно изменить строки для единого формата и избавиться от такого расхождения. 

### 25. Итоговый отчёт: для каждого класса профессии с числом вакансий не меньше 1 000 в каждом году покажите вакансии за 2019 и 2020, рост в процентах, медиану `net` за оба года, рост медианы и место по росту вакансий.

- **Таблицы:** vacancy_prof_classes, vacancies, v_salary
- **Что потренировать:** несколько CTE, соединение таблицы с собой, RANK()
- **Контроль:** 40 строк; лидер по росту: teacher, +65,8 %; devops: с 6 165 до 8 678 вакансий (+40,8 %), медиана с 115 000 до 135 000, 3-е место.

In [124]:
# Вопрос 25: q("""SELECT ...""")

# число вакансий по паре «класс + год»
# q("""                                                                                   
  # SELECT pc.prof_class,                                                                   
  #        extract(year FROM v.published_at) ::int AS yr,                                                                     
  #        count(*) AS vacancies                                                            
  # FROM vacancy_prof_classes pc                                                            
  # JOIN vacancies v ON v.vacancy_id = pc.vacancy_id                                                                 
  # GROUP BY pc.prof_class, yr                                                                           
  # ORDER BY pc.prof_class, yr;                                                             
#   """)  

# медиана net по паре «класс + год»
# q("""                                                                                   
#   SELECT pc.prof_class,                                                                   
#          s.year  AS yr,                                                                    
#          percentile_cont(0.5) WITHIN GROUP (ORDER BY s.net) AS median_net                   
#   FROM vacancy_prof_classes pc                                                            
#   JOIN v_salary s ON  s.vacancy_id = pc.vacancy_id                                                                
#   GROUP BY pc.prof_class, yr                                                                           
#   ORDER BY pc.prof_class, yr;                                                             
#   """)  

# General

q("""
WITH cnt AS (
  SELECT pc.prof_class,                                                                   
         extract(year FROM v.published_at) ::int AS yr,                                                                     
         count(*) AS vacancies                                                            
  FROM vacancy_prof_classes pc                                                            
  JOIN vacancies v ON v.vacancy_id = pc.vacancy_id                                                                 
  GROUP BY pc.prof_class, yr                                                                           
  ORDER BY pc.prof_class, yr       
),
med AS (
SELECT pc.prof_class,                                                                   
         s.year  AS yr,                                                                    
         percentile_cont(0.5) WITHIN GROUP (ORDER BY s.net) AS median_net                   
  FROM vacancy_prof_classes pc                                                            
  JOIN v_salary s ON  s.vacancy_id = pc.vacancy_id                                                                
  GROUP BY pc.prof_class, yr                                                                           
  ORDER BY pc.prof_class, yr 
),
yearly AS (                                                                             
      SELECT c.prof_class, c.yr, c.vacancies, m.median_net                                
      FROM cnt c                                                                          
      LEFT JOIN med m ON m.prof_class = c.prof_class                                      
                     AND m.yr         = c.yr                                              
  )                                                                                       
   SELECT a.prof_class,                                                                    
         a.vacancies     AS vac_2019,                                                     
         b.vacancies     AS vac_2020,                                                     
         round(100.0 * (b.vacancies - a.vacancies) / a.vacancies, 1)  AS vac_growth,                                               
         a.median_net    AS net_2019,                                                     
         b.median_net    AS net_2020,                                                     
         round((100.0 * (b.median_net - a.median_net) / a.median_net)::numeric, 1) AS net_growth,                                               
         RANK() OVER (ORDER BY (b.vacancies - a.vacancies)::numeric / a.vacancies DESC) AS growth_rank                                   
  FROM yearly a                                                                           
  JOIN yearly b ON b.prof_class = a.prof_class                                                                   
  WHERE a.yr = 2019                                                                       
    AND b.yr = 2020                                                                       
    AND a.vacancies >= 1000                                                               
    AND b.vacancies >= 1000                                                                                 
  ORDER BY growth_rank; 
""")


,prof_class,vac_2019,vac_2020,vac_growth,net_2019,net_2020,net_growth,growth_rank
0,teacher,1934,3207,65.8,40000.0,37500.0,-6.3,1
1,master,2283,3253,42.5,50000.0,45000.0,-10.0,2
2,devops,6165,8678,40.8,115000.0,135000.0,17.4,3
3,owner,1098,1517,38.2,121800.0,140000.0,14.9,4
4,data_scientist,3131,4118,31.5,120000.0,143550.0,19.6,5
5,teamlead,4218,5396,27.9,160000.0,180000.0,12.5,6
6,analyst,30700,38537,25.5,69600.0,70000.0,0.6,7
7,photo_video_prod,1157,1448,25.2,50000.0,55000.0,10.0,8
8,tester,20549,24767,20.5,60900.0,70000.0,14.9,9
9,montajnik,7710,9274,20.3,45000.0,50000.0,11.1,10


**Вывод:** 
 Из 52 классов профессий порог в 1 000 вакансий в каждом году прошли 40.               
                                                                                         
   Быстрее всего росло число вакансий учителей (+65,8 %) — вероятно, из-за перехода на   
   дистанционное обучение в 2020 году: школам и онлайн-школам понадобились преподаватели.
   Но медиана зарплаты у них упала на 6 %: спрос рос в основном на массовые и недорогие  
   позиции. Похожая картина у мастеров (+42,5 % вакансий, −10 % медианы).                
                                                                                         
   У IT-классов вырос и спрос, и зарплаты: devops (+40,8 % вакансий, медиана со 115 до   
   135 тыс., +17 %), data scientist (+31,5 %, +20 %), тимлиды (+28 %, +12,5 %). Это      
   согласуется с ускоренной цифровизацией в ковидный год. Рост и спроса, и цены — признак
   реального дефицита специалистов.                                                      
                                                                                         
   Рост числа вакансий и рост зарплаты — разные сигналы. Там, где растёт только число    
   вакансий при падающей медиане, скорее растёт массовый сегмент. Там, где растёт и то, и
   другое, работодатели конкурируют за кадры.                                            
                                                                                         
   Оговорка: число вакансий посчитано по всем вакансиям класса, а медиана — только по    
   рублёвым с зарплатой от 10 тыс. до 1 млн (v_salary). Это разные выборки. Кроме того,  
  ▎ одна вакансия может относиться к нескольким классам.  